# Evaluate Maximal Proper Subarchitectures of Known MFAs

This notebook has one purpose.

For every architecture already marked `is_minimal=True` in the selected CSV files, it forms every **maximal proper subarchitecture** obtained by decreasing exactly one hidden width by one:

\[
(d_0,\ldots,d_i,\ldots,d_L)
\longmapsto
(d_0,\ldots,d_i-1,\ldots,d_L),
\qquad 1\leq i\leq L-1.
\]

A candidate is ignored when the width being decreased is already \(1\). Candidates shared by multiple MFAs are evaluated only once.

For each architecture--exponent pair:

1. skip it when it already occurs in the CSV;
2. otherwise compute its dimension with `dim_backprop_gpu_only.py`;
3. append the result to the same CSV;
4. save periodically and save again before propagating a `KeyboardInterrupt`.

The notebook does **not** enumerate a predecessor box, search for new MFAs, or alter existing `is_minimal` flags.


## Configuration


In [ ]:
from pathlib import Path

# Make sure to edit MAX_NEW_EVALUATIONS_PER_FILE to None if you want to have a complete run

# Files
DATA_DIR = Path("Data")
FALLBACK_DATA_DIRS = [Path("data/raw"), Path("../data/raw")]

# None means every *_architectures.csv file in the selected data directory.
# Example: CSV_FILES = [Path("Data/2_1_architectures.csv")]
CSV_FILES = None

GPU_MODULE_PATH = None
GPU_MODULE_FILENAMES = ["dim_backprop_gpu_only.py"]

# Optional filters. None means use every recorded MFA.
H_VALUES = None
EXPONENTS = None
DEFAULT_EXPONENT = 2

# GPU dimension computation
PRIMES = (10_000_019, 19_511_957)
BASE_SEED = 20260630
RANK_WORKSPACE_BYTES = 512 * 1024**2
DIMENSION_VERBOSE = False

# Execution
RUN_FILL = True
MAX_NEW_EVALUATIONS_PER_FILE = 1_000
SAVE_EVERY_N_NEW_ROWS = 25

# A filling maximal proper subarchitecture contradicts minimality.
# The result is always saved before this optional stop.
STOP_ON_MFA_CONTRADICTION = False

CHECKPOINT_SUFFIX = ".maximal_subarchitecture_checkpoint"


## Imports and GPU backend


In [ ]:
import ast
import importlib.util
import json
import os
import sys
import time
import uuid
from collections import defaultdict
from typing import Sequence

import pandas as pd


def discover_gpu_module_path() -> Path:
    if GPU_MODULE_PATH is not None:
        path = Path(GPU_MODULE_PATH).expanduser().resolve()
        if not path.exists():
            raise FileNotFoundError(f"GPU_MODULE_PATH does not exist: {path}")
        return path

    checked = []
    for root in (Path.cwd(), Path.cwd().parent, Path("/mnt/data")):
        for filename in GPU_MODULE_FILENAMES:
            candidate = (root / filename).resolve()
            checked.append(candidate)
            if candidate.exists():
                return candidate

    checked_text = "\n".join(f"  - {path}" for path in checked)
    raise FileNotFoundError(
        "Could not find dim_backprop_gpu_only.py. Checked:\n" + checked_text
    )


def load_gpu_module(path: Path):
    module_name = "_mfa_dim_backprop_gpu_only"
    spec = importlib.util.spec_from_file_location(module_name, path)
    if spec is None or spec.loader is None:
        raise ImportError(f"Could not load the GPU module from {path}")

    module = importlib.util.module_from_spec(spec)
    sys.modules[module_name] = module
    spec.loader.exec_module(module)
    return module


GPU_MODULE_FILE = discover_gpu_module_path()
gpu_module = load_gpu_module(GPU_MODULE_FILE)
compute_dimension = gpu_module.compute_dimension
gpu_information = gpu_module.gpu_information

print("GPU module:", GPU_MODULE_FILE)
print("GPU information:", gpu_information())


## CSV and architecture helpers


In [ ]:
REQUIRED_COLUMNS = [
    "h",
    "exponent",
    "architecture",
    "num_parameters",
    "dimension_computed",
    "ambient_dimension",
    "is_full_dimension",
    "is_minimal",
]

AUDIT_COLUMNS = [
    "expected_dimension",
    "defect_expected",
    "defect_ambient",
    "backend",
    "primes",
    "elapsed_seconds",
    "status",
    "covered_by_mfa",
    "shrunk_hidden_layers",
]


def empty_architecture_dataframe() -> pd.DataFrame:
    """Return an empty dataframe with the complete expected schema."""
    return pd.DataFrame(
        {
            column: pd.Series(dtype="object")
            for column in REQUIRED_COLUMNS + AUDIT_COLUMNS
        }
    )


def ensure_expected_columns(df: pd.DataFrame) -> pd.DataFrame:
    """Add any columns needed by this notebook without deleting old columns."""
    df = df.copy()

    for column in REQUIRED_COLUMNS + AUDIT_COLUMNS:
        if column not in df.columns:
            df[column] = pd.Series(dtype="object")

    return df


def choose_data_dir() -> Path:
    if DATA_DIR.exists():
        return DATA_DIR

    for candidate in FALLBACK_DATA_DIRS:
        if candidate.exists():
            print(f"Using fallback data directory: {candidate}")
            return candidate

    return DATA_DIR


def find_csv_files(data_dir: Path) -> list[Path]:
    if CSV_FILES is not None:
        return [Path(path) for path in CSV_FILES]

    return sorted(data_dir.glob("*_architectures.csv"))


def checkpoint_path(path: Path) -> Path:
    return path.with_name(
        f"{path.stem}{CHECKPOINT_SUFFIX}{path.suffix}"
    )


def is_nonempty_file(path: Path) -> bool:
    """A valid CSV source must exist and contain at least one byte."""
    try:
        return path.is_file() and path.stat().st_size > 0
    except OSError:
        return False


def preferred_csv_source(path: Path) -> Path | None:
    """Choose the newest nonempty source.

    A zero-byte checkpoint is never allowed to override a valid main CSV.
    """
    checkpoint = checkpoint_path(path)

    main_valid = is_nonempty_file(path)
    checkpoint_valid = is_nonempty_file(checkpoint)

    if checkpoint_valid and (
        not main_valid
        or checkpoint.stat().st_mtime_ns > path.stat().st_mtime_ns
    ):
        print(
            f"Resuming {path.name} from newer checkpoint "
            f"{checkpoint.name}."
        )
        return checkpoint

    if main_valid:
        return path

    if checkpoint_valid:
        print(
            f"Using checkpoint {checkpoint.name} because "
            f"{path.name} is absent or empty."
        )
        return checkpoint

    return None


def read_architecture_csv(path: Path) -> pd.DataFrame:
    source = preferred_csv_source(path)

    if source is None:
        print(
            f"WARNING: {path} and its checkpoint are absent or empty. "
            "Treating this file as an empty table."
        )
        return empty_architecture_dataframe()

    try:
        df = pd.read_csv(source)

    except pd.errors.EmptyDataError:
        print(
            f"WARNING: {source} contains no readable CSV data. "
            "Treating it as an empty table."
        )
        return empty_architecture_dataframe()

    except pd.errors.ParserError as original_error:
        # Compatibility with older processed files having one preamble line.
        try:
            df = pd.read_csv(source, skiprows=1)
        except (pd.errors.EmptyDataError, pd.errors.ParserError) as fallback_error:
            raise RuntimeError(
                f"Could not parse {source} either normally or after "
                "skipping one preamble line."
            ) from fallback_error

    except (OSError, UnicodeError) as error:
        raise RuntimeError(f"Could not read CSV file {source}.") from error

    return ensure_expected_columns(df)


def save_csv(df: pd.DataFrame, path: Path) -> Path:
    """Save atomically, with a stable checkpoint if Windows locks the CSV."""
    path.parent.mkdir(parents=True, exist_ok=True)
    checkpoint = checkpoint_path(path)
    temporary = path.with_name(
        f".{path.name}.{os.getpid()}.{uuid.uuid4().hex}.tmp"
    )

    try:
        ensure_expected_columns(df).to_csv(temporary, index=False)

        try:
            os.replace(temporary, path)

            if checkpoint.exists():
                try:
                    checkpoint.unlink()
                except OSError:
                    pass

            return path

        except PermissionError:
            os.replace(temporary, checkpoint)
            print(
                f"WARNING: {path.name} is locked. "
                f"Saved the complete data to {checkpoint.name}."
            )
            return checkpoint

    finally:
        if temporary.exists():
            try:
                temporary.unlink()
            except OSError:
                pass


def parse_architecture(value) -> tuple[int, ...]:
    if isinstance(value, (tuple, list)):
        architecture = tuple(int(width) for width in value)
    else:
        if pd.isna(value):
            raise ValueError("Missing architecture.")

        architecture = tuple(
            int(width)
            for width in ast.literal_eval(str(value))
        )

    if len(architecture) < 2 or any(
        width <= 0 for width in architecture
    ):
        raise ValueError(f"Invalid architecture: {architecture}")

    return architecture


def architecture_string(
    architecture: Sequence[int],
) -> str:
    return str([int(width) for width in architecture])


def architecture_key(
    architecture: Sequence[int] | str,
    exponent: int,
) -> tuple[tuple[int, ...], int]:
    parsed = (
        parse_architecture(architecture)
        if isinstance(architecture, str)
        else tuple(int(width) for width in architecture)
    )

    return parsed, int(exponent)


def truthy(value) -> bool:
    if isinstance(value, bool):
        return value

    if pd.isna(value):
        return False

    return str(value).strip().lower() in {
        "true",
        "1",
        "yes",
        "y",
        "t",
    }


def selected(
    value: int,
    allowed: Sequence[int] | None,
) -> bool:
    return (
        allowed is None
        or int(value) in {int(item) for item in allowed}
    )


def parameter_count(
    architecture: Sequence[int],
) -> int:
    return sum(
        int(left) * int(right)
        for left, right in zip(
            architecture[:-1],
            architecture[1:],
        )
    )


def stable_seed(
    base_seed: int,
    architecture: Sequence[int],
    exponent: int,
) -> int:
    value = int(base_seed) + 107 * int(exponent)

    for index, width in enumerate(architecture):
        value += (index + 1) * 1_000_003 * int(width)

    return value % (2**31 - 1)


def existing_architecture_keys(
    df: pd.DataFrame,
) -> set[tuple[tuple[int, ...], int]]:
    keys = set()

    for _, row in df.iterrows():
        try:
            exponent_value = row.get("exponent")
            exponent = (
                DEFAULT_EXPONENT
                if pd.isna(exponent_value)
                else int(exponent_value)
            )

            keys.add(
                architecture_key(
                    row["architecture"],
                    exponent,
                )
            )

        except Exception:
            continue

    return keys


## Build the maximal-subarchitecture targets


In [ ]:
def maximal_proper_subarchitectures(
    architecture: Sequence[int],
) -> list[tuple[tuple[int, ...], int]]:
    """Return (candidate, hidden-layer index) for every one-step shrink."""
    parent = tuple(int(width) for width in architecture)
    targets = []

    for layer in range(1, len(parent) - 1):
        if parent[layer] <= 1:
            continue
        candidate = list(parent)
        candidate[layer] -= 1
        targets.append((tuple(candidate), layer))

    return targets


def collect_targets(df: pd.DataFrame) -> dict:
    """Deduplicate one-step subarchitectures of all selected recorded MFAs."""
    targets = defaultdict(
        lambda: {
            "parent_mfas": set(),
            "shrunk_hidden_layers": set(),
        }
    )

    for row_index, row in df.iterrows():
        if not truthy(row.get("is_minimal")):
            continue

        parent = parse_architecture(row["architecture"])
        h = len(parent) - 1
        exponent_value = row.get("exponent")
        exponent = (
            DEFAULT_EXPONENT
            if pd.isna(exponent_value)
            else int(exponent_value)
        )

        if not selected(h, H_VALUES) or not selected(exponent, EXPONENTS):
            continue

        if not truthy(row.get("is_full_dimension")):
            print(
                f"Warning: row {row_index} is marked minimal but not filling. "
                "It is still used because is_minimal=True is the source of truth."
            )

        for candidate, layer in maximal_proper_subarchitectures(parent):
            key = (candidate, exponent)
            targets[key]["parent_mfas"].add(parent)
            targets[key]["shrunk_hidden_layers"].add(layer)

    return dict(targets)


def make_plan(df: pd.DataFrame) -> pd.DataFrame:
    existing = existing_architecture_keys(df)
    targets = collect_targets(df)

    rows = []
    for (candidate, exponent), metadata in sorted(
        targets.items(),
        key=lambda item: (
            item[0][1],
            len(item[0][0]),
            parameter_count(item[0][0]),
            item[0][0],
        ),
    ):
        rows.append(
            {
                "architecture": architecture_string(candidate),
                "exponent": exponent,
                "h": len(candidate) - 1,
                "already_in_csv": (candidate, exponent) in existing,
                "parent_mfas": json.dumps(
                    [list(parent) for parent in sorted(metadata["parent_mfas"])]
                ),
                "shrunk_hidden_layers": json.dumps(
                    sorted(metadata["shrunk_hidden_layers"])
                ),
            }
        )

    return pd.DataFrame(rows)


## Evaluate missing targets and append them


In [ ]:
def evaluate_target(
    architecture: tuple[int, ...],
    exponent: int,
    parent_mfas: set[tuple[int, ...]],
    shrunk_hidden_layers: set[int],
) -> dict:
    seed = stable_seed(BASE_SEED, architecture, exponent)
    start = time.perf_counter()

    result = compute_dimension(
        architecture,
        exponent,
        primes=PRIMES,
        seed=seed,
        rank_workspace_bytes=RANK_WORKSPACE_BYTES,
        verbose=DIMENSION_VERBOSE,
    )
    elapsed = time.perf_counter() - start

    (
        returned_architecture,
        returned_exponent,
        ambient_dimension,
        expected_dimension,
        dimension,
        expected_defect,
    ) = result

    returned_architecture = tuple(int(width) for width in returned_architecture)
    if returned_architecture != architecture:
        raise RuntimeError(
            f"Backend returned {returned_architecture}, expected {architecture}."
        )
    if int(returned_exponent) != int(exponent):
        raise RuntimeError(
            f"Backend returned exponent {returned_exponent}, expected {exponent}."
        )

    is_full = int(dimension) == int(ambient_dimension)

    return {
        "h": len(architecture) - 1,
        "exponent": int(exponent),
        "architecture": architecture_string(architecture),
        "num_parameters": parameter_count(architecture),
        "dimension_computed": int(dimension),
        "ambient_dimension": int(ambient_dimension),
        "is_full_dimension": bool(is_full),
        "is_minimal": False,
        "expected_dimension": int(expected_dimension),
        "defect_expected": int(expected_defect),
        "defect_ambient": int(ambient_dimension) - int(dimension),
        "backend": "dim_backprop_gpu_only/CuPy-CUDA",
        "primes": str(tuple(int(prime) for prime in PRIMES)),
        "elapsed_seconds": elapsed,
        "status": (
            "filling_maximal_subarchitecture_contradiction"
            if is_full
            else "maximal_subarchitecture_of_recorded_mfa"
        ),
        "covered_by_mfa": json.dumps(
            [list(parent) for parent in sorted(parent_mfas)]
        ),
        "shrunk_hidden_layers": json.dumps(
            sorted(int(layer) for layer in shrunk_hidden_layers)
        ),
    }


def append_record(df: pd.DataFrame, record: dict) -> pd.DataFrame:
    for column in record:
        if column not in df.columns:
            df[column] = pd.Series(dtype="object")

    new_row = {column: pd.NA for column in df.columns}
    new_row.update(record)
    return pd.concat([df, pd.DataFrame([new_row])], ignore_index=True)


def fill_one_csv(
    path: Path,
    *,
    max_new_evaluations: int | None = None,
) -> dict:
    print("=" * 100)
    print(f"Processing {path}")
    print("=" * 100)

    df = read_architecture_csv(path)
    existing = existing_architecture_keys(df)
    targets = collect_targets(df)

    ordered_targets = sorted(
        targets.items(),
        key=lambda item: (
            item[0][1],
            len(item[0][0]),
            parameter_count(item[0][0]),
            item[0][0],
        ),
    )
    missing_targets = [
        (key, metadata)
        for key, metadata in ordered_targets
        if key not in existing
    ]

    print(f"Recorded one-step targets: {len(ordered_targets):,}")
    print(f"Already present: {len(ordered_targets) - len(missing_targets):,}")
    print(f"Missing evaluations: {len(missing_targets):,}")

    new_evaluations = 0
    contradictions = []
    unsaved_rows = 0

    try:
        for position, ((architecture, exponent), metadata) in enumerate(
            missing_targets,
            start=1,
        ):
            if (
                max_new_evaluations is not None
                and new_evaluations >= int(max_new_evaluations)
            ):
                print(
                    "Reached MAX_NEW_EVALUATIONS_PER_FILE="
                    f"{int(max_new_evaluations)}."
                )
                break

            print(
                f"[{position:,}/{len(missing_targets):,}] "
                f"Evaluating {architecture}, r={exponent} ..."
            )

            record = evaluate_target(
                architecture,
                exponent,
                metadata["parent_mfas"],
                metadata["shrunk_hidden_layers"],
            )
            df = append_record(df, record)
            existing.add((architecture, exponent))
            new_evaluations += 1
            unsaved_rows += 1

            print(
                f"  dim={record['dimension_computed']}/"
                f"{record['ambient_dimension']} | "
                f"{record['status']} | "
                f"{record['elapsed_seconds']:.2f}s"
            )

            if (
                SAVE_EVERY_N_NEW_ROWS is not None
                and int(SAVE_EVERY_N_NEW_ROWS) > 0
                and unsaved_rows >= int(SAVE_EVERY_N_NEW_ROWS)
            ):
                save_csv(df, path)
                unsaved_rows = 0

            if record["is_full_dimension"]:
                contradictions.append(architecture)
                save_csv(df, path)
                unsaved_rows = 0
                message = (
                    f"{architecture} is a filling maximal proper subarchitecture "
                    "of a recorded MFA. The corresponding is_minimal=True flag "
                    "is contradicted."
                )
                if STOP_ON_MFA_CONTRADICTION:
                    raise RuntimeError(message)
                print("WARNING:", message)

    except KeyboardInterrupt:
        print("\nKeyboardInterrupt received. Saving completed evaluations ...")
        save_csv(df, path)
        raise
    finally:
        if unsaved_rows:
            save_csv(df, path)

    save_csv(df, path)

    return {
        "path": str(path),
        "targets": len(ordered_targets),
        "already_present": len(ordered_targets) - len(missing_targets),
        "missing_before_run": len(missing_targets),
        "new_evaluations": new_evaluations,
        "filling_contradictions": [
            architecture_string(architecture)
            for architecture in contradictions
        ],
    }


## Inspect the plan


In [ ]:
data_dir = choose_data_dir()
csv_paths = find_csv_files(data_dir)

In [ ]:
import random
# random.shuffle(csv_paths)

In [ ]:
print("Data directory:", data_dir.resolve())
print("CSV files:")
for path in csv_paths:
    print(" -", path)


In [ ]:
if not csv_paths:
    raise FileNotFoundError(
        f"No *_architectures.csv files were found in {data_dir}. "
        "Set DATA_DIR or CSV_FILES in the configuration cell."
    )

plan_frames = []
for path in csv_paths:
    file_plan = make_plan(read_architecture_csv(path))
    if not file_plan.empty:
        file_plan.insert(0, "file", str(path))
        plan_frames.append(file_plan)

plan_df = (
    pd.concat(plan_frames, ignore_index=True)
    if plan_frames
    else pd.DataFrame()
)

display(plan_df)

if not plan_df.empty:
    print("\nPlan counts:")
    display(
        plan_df.groupby(["file", "already_in_csv"])
        .size()
        .rename("count")
        .reset_index()
    )


In [ ]:
if not csv_paths:
    raise FileNotFoundError(
        f"No *_architectures.csv files were found in {data_dir}. "
        "Set DATA_DIR or CSV_FILES in the configuration cell."
    )

plan_frames = []
for path in csv_paths:
    file_plan = make_plan(read_architecture_csv(path))
    if not file_plan.empty:
        file_plan.insert(0, "file", str(path))
        plan_frames.append(file_plan)

plan_df = (
    pd.concat(plan_frames, ignore_index=True)
    if plan_frames
    else pd.DataFrame()
)

display(plan_df)

if not plan_df.empty:
    print("\nPlan counts:")
    display(
        plan_df.groupby(["file", "already_in_csv"])
        .size()
        .rename("count")
        .reset_index()
    )

## Run


In [ ]:
summaries = []

if RUN_FILL:
    for csv_path in csv_paths:
        summaries.append(
            fill_one_csv(
                csv_path,
                max_new_evaluations=MAX_NEW_EVALUATIONS_PER_FILE,
            )
        )
else:
    print("RUN_FILL=False. Review plan_df and then enable the run.")

summary_df = pd.DataFrame(summaries)
display(summary_df)


## Verify that no selected one-step targets remain absent


In [ ]:
remaining_rows = []

for path in csv_paths:
    df = read_architecture_csv(path)
    existing = existing_architecture_keys(df)
    targets = collect_targets(df)

    for candidate, exponent in targets:
        if (candidate, exponent) not in existing:
            remaining_rows.append(
                {
                    "file": str(path),
                    "architecture": architecture_string(candidate),
                    "exponent": exponent,
                }
            )

remaining_df = pd.DataFrame(remaining_rows)
if remaining_df.empty:
    print("Every selected MFA maximal proper subarchitecture is present.")
else:
    print("Some targets remain absent:")
    display(remaining_df)
